In [ ]:
#!pip install --upgrade transformers

In [21]:
from transformers import pipeline
import json
from pathlib import Path

import pandas as pd

In [22]:
import transformers
print(transformers.__version__)

5.4.0


In [23]:
# ── 1. Load model ─────────────────────────────────────────────────────────────
# Good small options to try:
# - "mistralai/Mistral-7B-Instruct-v0.3"
# - "Qwen/Qwen2.5-7B-Instruct"
# - "BioMistral/BioMistral-7B" (medical-focused)

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

pipe = pipeline(
    "text-generation",
    model=MODEL_NAME,
    device_map="auto",      # automatically uses GPU if available, else CPU
    max_new_tokens=1024,
)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [24]:

# ── 2. Prompt ─────────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are a medical information extraction assistant.
Given a medical text, extract structured information and return it as JSON only,
with no preamble or explanation.
"""

def build_prompt(text: str) -> str:
    return f"""Extract the following fields from the medical text below.
If a field is not mentioned, use null.

Fields to extract:
- symptoms: list of symptoms reported by the patient
- diagnosis: the diagnosis if mentioned
- medications: list of medications mentioned
- patient_age: age of the patient if mentioned
- patient_gender: gender of the patient if mentioned

Return only a JSON object with these fields.

Medical text:
{text}
"""


In [25]:

# ── 3. Extraction function ────────────────────────────────────────────────────
def extract_medical_info(text: str) -> dict:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": build_prompt(text)}
    ]
    output = pipe(messages)
    raw = output[0]["generated_text"][-1]["content"].strip()

    # Strip markdown code fences if present
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]

    try:
        return json.loads(raw.strip())
    except json.JSONDecodeError:
        # Return raw text if JSON parsing fails, so you can inspect it
        print(f"Warning: could not parse JSON, returning raw output.")
        return {"raw_output": raw}



In [26]:
# ── 4. Process a folder of .txt files ────────────────────────────────────────
# Test on just the first 5 files
def process_er_folder(folder_path: str, limit: int = None) -> pd.DataFrame:
    records = []
    txt_files = list(Path(folder_path).glob("*.txt"))
    if limit:
        txt_files = txt_files[:limit]
    print(f"Processing {len(txt_files)} files...")

    for i, txt_file in enumerate(txt_files):
        print(f"[{i+1}/{len(txt_files)}] {txt_file.name}...")
        text = txt_file.read_text(encoding="utf-8").strip()
        if not text:
            continue
        extracted = extract_medical_info(text)
        extracted["source_file"] = txt_file.name
        records.append(extracted)

    return pd.DataFrame(records)


In [27]:
# Try 5 first
df_test = process_er_folder("mtsamples_er/texts", limit=5)
print(df_test)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processing 5 files...
[1/5] Sample_Name Abdominal_Pain_-_Consult
Medical_Specialty 
			__
				Emergency_Room_Reports.txt...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[2/5] Sample_Name Abrasions_&_Lacerations_-_ER_Visit
Medical_Specialty 
			__
				Emergency_Room_Reports.txt...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[3/5] Sample_Name Accidental_Celesta_Ingestion_-_ER_Visit
Medical_Specialty 
			__
				Emergency_Room_Reports.txt...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[4/5] Sample_Name Acute_Inferior_Myocardial_Infarction
Medical_Specialty 
			__
				Emergency_Room_Reports.txt...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[5/5] Sample_Name Agitation_-_ER_Visit
Medical_Specialty 
			__
				Emergency_Room_Reports.txt...
                                            symptoms  \
0  [abdominal pain, persistent, anorexia, obstipa...   
1  [Abrasions & Lacerations, neck pain, loss of c...   
2                     [Accidental Celesta Ingestion]   
3  [chest pain, shortness of breath, diaphoresis,...   
4  [Agitation, Dry heaves, Feeling poisoned, Daug...   

                                           diagnosis patient_age  \
0                             sigmoid diverticulitis        None   
1  [Concussion, Facial abrasion, Scalp laceration...        None   
2                                               None           3   
3               Acute Inferior Myocardial Infarction        None   
4                                               None        None   

  patient_gender                                        medications  \
0           None                                    [Cipro, Flagyl]   
1           No

In [28]:
# Or look at one extracted result
print(df_test.iloc[0].to_dict())

{'symptoms': ['abdominal pain', 'persistent', 'anorexia', 'obstipation', 'no nausea or vomiting', 'passing flatus', 'no bright red blood per rectum', 'no history of recent melena', 'no definite fevers or chills', 'no history of jaundice', 'no significant recent weight loss'], 'diagnosis': 'sigmoid diverticulitis', 'patient_age': None, 'patient_gender': None, 'medications': ['Cipro', 'Flagyl'], 'source_file': 'Sample_Name\xa0Abdominal_Pain_-_Consult\nMedical_Specialty\xa0\n\t\t\t__\n\t\t\t\tEmergency_Room_Reports.txt'}


In [8]:
#!du -sh ~/.cache/huggingface/hub/

256G	/home/tatiana.bladier/.cache/huggingface/hub/
